# 🟡 Тренажер по NumPy: Уровень 2 (Продвинутый + ML-собеседования)

Как работать с этим ноутбуком:
1. Внимательно прочитай блок **«Методы для задачи»** и **«Связанные методы в копилку»** перед заданием.
2. Замени `pass` в кодовой ячейке на свое решение (никаких циклов `for`, только векторизация!).
3. Запусти ячейку (`Shift + Enter`), чтобы проверить себя на `assert`-тестах.
4. Если что-то не получается — открой спойлер с подсказкой и готовым решением.

In [1]:
# Запусти эту ячейку один раз перед началом работы!
import numpy as np
print("✅ Библиотека NumPy успешно импортирована! Версия:", np.__version__)

✅ Библиотека NumPy успешно импортирована! Версия: 2.5.2


---
## Модуль 1. Broadcasting (Транслирование) и `keepdims=True`

### 🎯 Правила и методы, которые нужны для задачи:
* **Broadcasting** позволяет выполнять арифметические операции над массивами разных форм. NumPy сравнивает размерности массивов **справа налево**: две оси совместимы, если они равны ИЛИ одна из них равна `1`.
* Например, матрица `(N, M)` и вектор `(M,)` (или `(1, M)`) совместимы: вектор автоматически «растянется» на все `N` строк.
* А вот матрица `(N, M)` и вектор длины `N` формы `(N,)` напрямую **не вычитаются** (справа стоит `M` и `N`). Чтобы вычесть вектор из каждого столбца (построчно), ему нужно сделать форму `(N, 1)`.
* Параметр **`keepdims=True`** в агрегациях (`np.mean`, `np.std`, `np.sum`, `np.max`) сохраняет схлопнутую ось как размерность длины `1`:
  * Если `X` имеет форму `(3, 4)`, то `np.mean(X, axis=1, keepdims=True)` вернет столбец `(3, 1)`, который сразу можно вычитать из `X`!

### 📚 Связанные методы в копилку:
* `np.broadcast_to(arr, new_shape)` — явно размножает массив до новой формы (возвращает read-only view без копирования памяти).
* В ML стандартизация признаков (StandardScaler) по столбцам записывается в одну строчку: `(X - X.mean(axis=0)) / X.std(axis=0)`.

### Задача 1
Напиши функцию `normalize_rows_and_cols(X)`, которая принимает 2D-матрицу объектов и признаков `X` размера `(N, M)` и возвращает кортеж из двух матриц `(row_centered, col_standardized)`:
1. `row_centered` — матрица `X`, в которой из **каждой строки** вычтено среднее арифметическое этой же строки (используй `axis=1, keepdims=True`).
2. `col_standardized` — матрица `X`, в которой каждый **столбец** стандартизирован (Z-score): из каждого столбца вычтено его среднее и результат поделен на стандартное отклонение этого столбца (`np.std` по `axis=0`). Гарантируется, что стандартное отклонение не равно нулю.

In [15]:
arr = np.array([[1,2],[3,4]])
arr - np.mean(arr,axis=1,keepdims=True)

array([[-0.5,  0.5],
       [-0.5,  0.5]])

In [17]:
def normalize_rows_and_cols(X: np.ndarray):
    # ТВОЙ КОД ЗДЕСЬ
    return (X - np.mean(X,axis=1,keepdims=True), (X - np.mean(X,axis=0)) / np.std(X,axis=0))
    # pass


# === ТЕСТЫ ===
mat = np.array([
    [1.0, 2.0, 3.0],
    [10.0, 20.0, 30.0]
])
r_cent, c_std = normalize_rows_and_cols(mat)
expected_r = np.array([[-1.0, 0.0, 1.0], [-10.0, 0.0, 10.0]])
expected_c = np.array([[-1.0, -1.0, -1.0], [1.0, 1.0, 1.0]])
assert np.allclose(r_cent, expected_r), f"Ошибка в центрировании строк:\n{r_cent}"
assert np.allclose(c_std, expected_c), f"Ошибка в стандартизации столбцов:\n{c_std}"
print("✅ Задача 1 решена верно!")

✅ Задача 1 решена верно!


---
## Модуль 2. Создание фиктивных осей (`None` / `np.newaxis`) и попарные операции

### 🎯 Синтаксис, который нужен для задачи:
* Индексация с помощью `None` (или `np.newaxis`) вставляет новую ось длины `1` на нужное место:
  * Если `a` имеет форму `(N,)`, то `a[:, None]` — это столбец формы `(N, 1)`.
  * Если `b` имеет форму `(M,)`, то `b[None, :]` — это строка формы `(1, M)`.
* **Магия попарных операций:** если вычесть или сложить столбец `(N, 1)` и строку `(1, M)`, Broadcasting растянет оба массива до матрицы **`(N, M)`**, где на пересечении `(i, j)` будет результат операции между `a[i]` и `b[j]`!

### 📚 Связанные методы в копилку:
* `np.outer(a, b)` — внешнее произведение векторов (эквивалентно `a[:, None] * b[None, :]`).
* `np.subtract.outer(a, b)` / `np.add.outer(a, b)` — встроенный способ получить матрицу попарных разностей или сумм для 1D-массивов.
* Если нужно посчитать попарные расстояния между $N$ точками в $D$-мерном пространстве (матрицы `(N, D)` и `(M, D)`), трюк с `None` работает так: `X[:, None, :] - Y[None, :, :]` дает тензор `(N, M, D)`!

### Задача 2
Даны два одномерных массива координат точек на прямой: `x` (длины `N`) и `y` (длины `M`).

Напиши функцию `pairwise_abs_diff(x, y)`, которая **без циклов** возвращает 2D-матрицу `D` размера `(N, M)`, где элемент `D[i, j] = |x[i] - y[j]|` (модуль разности $i$-го элемента `x` и $j$-го элемента `y`).

In [21]:
def pairwise_abs_diff(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    # ТВОЙ КОД ЗДЕСЬ
    return np.abs(x[:,None] - y[None,:])
    # pass


# === ТЕСТЫ ===
x_pts = np.array([1, 5, 10])
y_pts = np.array([2, 4])
dist_mat = pairwise_abs_diff(x_pts, y_pts)
expected_dist = np.array([
    [1, 3],
    [3, 1],
    [8, 6]
])
assert dist_mat.shape == (3, 2), f"Неверная форма: {dist_mat.shape}"
assert np.array_equal(dist_mat, expected_dist), f"Ошибка в матрице расстояний:\n{dist_mat}"
print("✅ Задача 2 решена верно!")

✅ Задача 2 решена верно!


---
## Модуль 3. Работа с пропусками и бесконечностями (`NaN` и `Inf`)

### 🎯 Методы, которые нужны для задачи:
* В вещественных массивах пропуски обозначаются как `np.nan` (Not a Number). **Важно:** `np.nan == np.nan` всегда возвращает `False`! Проверять на `NaN` через `==` нельзя.
* `np.isnan(arr)` — возвращает булеву маску (`True` там, где стоит `np.nan`).
* Обычные функции `np.sum` или `np.mean` при наличии хотя бы одного `NaN` возвращают `NaN`. Чтобы игнорировать пропуски при подсчете статистик, используй семейство **`nan`-функций**:
  * `np.nanmean(arr, axis=...)` — среднее без учета `NaN`.
  * `np.nansum(arr, axis=...)`, `np.nanmax(...)`, `np.nanstd(...)`.
* `np.where(np.isnan(X), replacement, X)` — элегантный способ заменить все `NaN` в матрице `X` на соответствующие значения из `replacement` (с поддержкой Broadcasting!).

### 📚 Связанные методы в копилку:
* `np.isinf(arr)` — проверяет элементы на бесконечность (`+np.inf` или `-np.inf`).
* `np.isfinite(arr)` — возвращает `True` для обычных чисел (не `NaN` и не `Inf`).
* `np.nan_to_num(arr, nan=0.0, posinf=1e10, neginf=-1e10)` — быстрая замена всех `NaN` и `Inf` на константы одной командой.

### Задача 3
Частая задача в предобработке данных (SimpleImputer) — заполнить пропуски в таблице средним значением по каждому признаку (столбцу).

Напиши функцию `impute_column_means(X)`, которая принимает 2D-матрицу `X` с пропусками `np.nan` и возвращает новую матрицу, где каждый `np.nan` заменен на **среднее значение того столбца**, в котором он находился (гарантируется, что в каждом столбце есть хотя бы одно не-NaN число).

In [22]:
def impute_column_means(X: np.ndarray) -> np.ndarray:
    # ТВОЙ КОД ЗДЕСЬ
    return np.where(np.isnan(X),np.nanmean(X,axis=0),X)
    # pass


# === ТЕСТЫ ===
raw_data = np.array([
    [1.0, np.nan, 10.0],
    [3.0, 20.0, np.nan],
    [np.nan, 40.0, 30.0]
])
imputed = impute_column_means(raw_data)
expected_imp = np.array([
    [1.0, 30.0, 10.0],
    [3.0, 20.0, 20.0],
    [2.0, 40.0, 30.0]
])
assert not np.isnan(imputed).any(), "В матрице остались значения NaN!"
assert np.allclose(imputed, expected_imp), f"Неверное заполнение пропусков:\n{imputed}"
print("✅ Задача 3 решена верно!")

✅ Задача 3 решена верно!


---
## Модуль 4. Fancy Indexing (Прихотливая индексация) и One-Hot Encoding

### 🎯 Синтаксис, который нужен для задачи:
* В качестве индексов в квадратные скобки можно передавать **списки или целочисленные массивы индексов**:
  * `arr[[0, 4, 2]]` — достанет 0-й, 4-й и 2-й элементы именно в таком порядке.
  * Для 2D-матрицы `mat[row_indices, col_indices]` достанет элементы с координатами `(row_indices[k], col_indices[k])`.
* **Классический трюк для One-Hot Encoding:**
  1. Способ через присваивание: создаем матрицу нулей `zeros = np.zeros((N, num_classes))` и ставим единицы в нужные столбцы каждой строки: `zeros[np.arange(N), labels] = 1`.
  2. Способ в одну строку через `np.eye`: если взять единичную матрицу `np.eye(num_classes)` и проиндексировать её строками из вектора меток `labels` (`np.eye(num_classes)[labels]`), получится готовая One-Hot матрица!

### 📚 Связанные методы в копилку:
* `np.take_along_axis(arr, indices, axis=...)` — удобная функция для выборки элементов по индексам вдоль конкретной оси.
* `np.add.at(arr, indices, values)` — прибавляет значения по индексам корректно, даже если в `indices` есть повторяющиеся индексы (обычное `arr[indices] += 1` при дубликатах прибавит единицу только один раз из-за буферизации!).

### Задача 4
Напиши функцию `to_one_hot(labels, num_classes)`, которая принимает 1D-массив целочисленных меток классов `labels` (числа от `0` до `num_classes - 1`) длины `N` и возвращает 2D-матрицу One-Hot кодирования размера `(N, num_classes)` типа `float`.

In [24]:
def to_one_hot(labels: np.ndarray, num_classes: int) -> np.ndarray:
    # ТВОЙ КОД ЗДЕСЬ
    zeros = np.zeros((len(labels), num_classes))
    zeros[np.arange(len(labels)), labels] = 1
    return zeros
    # pass


# === ТЕСТЫ ===
y = np.array([2, 0, 1, 2])
ohe = to_one_hot(y, num_classes=3)
expected_ohe = np.array([
    [0., 0., 1.],
    [1., 0., 0.],
    [0., 1., 0.],
    [0., 0., 1.]
])
assert ohe.shape == (4, 3), f"Неверная форма: {ohe.shape}"
assert np.array_equal(ohe, expected_ohe), f"Неверный One-Hot Encoding:\n{ohe}"
print("✅ Задача 4 решена верно!")

✅ Задача 4 решена верно!


In [ ]:
def to_one_hot(labels: np.ndarray, num_classes: int) -> np.ndarray:
    # ТВОЙ КОД ЗДЕСЬ
    return np.eye(num_classes)[labels]
    # pass


# === ТЕСТЫ ===
y = np.array([2, 0, 1, 2])
ohe = to_one_hot(y, num_classes=3)
expected_ohe = np.array([
    [0., 0., 1.],
    [1., 0., 0.],
    [0., 1., 0.],
    [0., 0., 1.]
])
assert ohe.shape == (4, 3), f"Неверная форма: {ohe.shape}"
assert np.array_equal(ohe, expected_ohe), f"Неверный One-Hot Encoding:\n{ohe}"
print("✅ Задача 4 решена верно!")

✅ Задача 4 решена верно!


---
## Модуль 5. Продвинутая сортировка (`np.argsort`) и поиск Top-K

### 🎯 Методы, которые нужны для задачи:
* `np.argsort(arr, axis=-1)` — возвращает массив **индексов**, которые отсортировали бы массив по возрастанию.
* Как получить сортировку **по убыванию**:
  * В 1D-массиве: `np.argsort(arr)[::-1]` или `np.argsort(-arr)` (для численных массивов).
  * В 2D-матрице по строкам (по убыванию): `np.argsort(-mat, axis=1)` или `np.argsort(mat, axis=1)[:, ::-1]`.

### 📚 Связанные методы в копилку:
* `np.argpartition(arr, kth)` — находит индексы Top-K элементов за линейное время $O(N)$ (алгоритм Quickselect) без полной сортировки всего массива за $O(N \log N)$!
* Сортировка 2D-таблицы по конкретному $k$-му столбцу: `mat[mat[:, k].argsort()]`.

### Задача 5
В рекомендательных системах и классификации нужно уметь находить `k` самых вероятных классов для каждого объекта.

Напиши функцию `get_top_k_indices(probs, k)`, которая принимает 2D-матрицу вероятностей `probs` размера `(N_samples, M_classes)` и возвращает 2D-матрицу размера `(N_samples, k)`, где в каждой строке лежат **индексы `k` самых больших значений** этой строки, упорядоченные **по убыванию** вероятности.

In [43]:
p = np.array([
    [0.15, 0.50, 0.25, 0.10],
    [0.40, 0.10, 0.05, 0.45]
])
k=2
np.argsort(p,axis=1)[:,::-1]


array([[1, 2, 0, 3],
       [3, 0, 1, 2]])

In [41]:
def get_top_k_indices(probs: np.ndarray, k: int) -> np.ndarray:
    # ТВОЙ КОД ЗДЕСЬ
    return np.argsort(probs,axis=1)[:,::-1][:,:k]
    # pass


# === ТЕСТЫ ===
p = np.array([
    [0.1, 0.5, 0.2, 0.2],  # Топ-2 индекса: 1 (0.5), затем 2 или 3. Пусть будут все разные:
    [0.1, 0.6, 0.2, 0.1]
])
p = np.array([
    [0.15, 0.50, 0.25, 0.10],
    [0.40, 0.10, 0.05, 0.45]
])
top2 = get_top_k_indices(p, k=2)
expected_top2 = np.array([
    [1, 2],
    [3, 0]
])
assert top2.shape == (2, 2), f"Неверная форма: {top2.shape}"
assert np.array_equal(top2, expected_top2), f"Неверные Top-K индексы:\n{top2}"
print("✅ Задача 5 решена верно!")

✅ Задача 5 решена верно!


---
## Модуль 6. Линейная алгебра (`np.linalg`) и L2-нормализация

### 🎯 Методы, которые нужны для задачи:
* `np.linalg.norm(arr, ord=None, axis=None, keepdims=False)` — вычисляет норму (длину) вектора или матрицы.
  * По умолчанию считает стандартную **Евклидову ($L_2$) норму**: $\|x\|_2 = \sqrt{\sum x_i^2}$.
  * Если указать `axis=1, keepdims=True`, посчитает длину каждой строки матрицы как вектор-столбец `(N, 1)`.
* `np.linalg.solve(A, b)` — решает систему линейных алгебраических уравнений $Ax = b$ (работает быстрее и точнее, чем поиск обратной матрицы `np.linalg.inv(A) @ b`!).

### 📚 Связанные методы в копилку:
* `np.linalg.inv(A)` — обратная матрица $A^{-1}$ (для вырожденных матриц используй псевдообратную `np.linalg.pinv(A)`).
* `np.linalg.det(A)` — определитель матрицы.
* `np.linalg.eig(A)` / `np.linalg.eigh(A)` — собственные значения и собственные векторы (для симметричных матриц — `eigh`).
* `np.linalg.svd(A)` — сингулярное разложение матрицы (SVD).

### Задача 6
Напиши функцию `l2_normalize_and_solve(vectors, A, b)`, которая:
1. Принимает 2D-матрицу `vectors` размера `(N, D)` и нормирует каждую её строку на единичную $L_2$-длину (делит каждую строку на её $L_2$-норму). Гарантируется, что нулевых строк нет.
2. Принимает невырожденную квадратную матрицу `A` размера `(M, M)` и вектор `b` длины `M`, и находит вектор решения системы $Ax = b$.
3. Возвращает кортеж `(unit_vectors, x_solution)`.

In [46]:
def l2_normalize_and_solve(vectors: np.ndarray, A: np.ndarray, b: np.ndarray):
    # ТВОЙ КОД ЗДЕСЬ
    return (vectors / np.linalg.norm(vectors,axis=1) , np.linalg.solve(A,b))
    # pass


# === ТЕСТЫ ===
vecs = np.array([
    [3.0, 4.0],
    [0.0, 5.0]
])
A_mat = np.array([[2.0, 1.0], [1.0, 3.0]])
b_vec = np.array([5.0, 5.0])  # Решение: x = [2.0, 1.0]

u_vecs, x_sol = l2_normalize_and_solve(vecs, A_mat, b_vec)
assert np.allclose(u_vecs, np.array([[0.6, 0.8], [0.0, 1.0]])), f"Неверная L2-нормализация:\n{u_vecs}"
assert np.allclose(x_sol, np.array([2.0, 1.0])), f"Неверное решение СЛАУ: {x_sol}"
print("✅ Задача 6 решена верно!")

✅ Задача 6 решена верно!


<details>
<summary>💡 <b>Показать подсказку и решение (Задача 6)</b></summary>

Подсказка: Используй `np.linalg.norm(vectors, axis=1, keepdims=True)` для подсчета норм строк и `np.linalg.solve(A, b)` для СЛАУ.

```python
def l2_normalize_and_solve(vectors: np.ndarray, A: np.ndarray, b: np.ndarray):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    unit_vectors = vectors / norms
    x_solution = np.linalg.solve(A, b)
    return unit_vectors, x_solution
```
</details>

---
## Модуль 7. Диагонали, след и треугольные матрицы

### 🎯 Методы, которые нужны для задачи:
* `np.diag(v)` — работает в обе стороны:
  * Если передать **2D-матрицу**, извлечет её главную диагональ в виде 1D-вектора.
  * Если передать **1D-вектор**, построит диагональную 2D-матрицу с этим вектором на диагонали!
* `np.trace(mat)` — возвращает след матрицы (сумму элементов на главной диагонали).
* `np.triu(mat, k=0)` — возвращает верхнетреугольную часть матрицы (всё, что ниже диагонали `k`, зануляется). При `k=1` зануляется и сама главная диагональ!
* `np.tril(mat, k=0)` — возвращает нижнетреугольную часть матрицы.

### 📚 Связанные методы в копилку:
* `np.fill_diagonal(mat, val)` — изменяет главную диагональ матрицы **на месте (in-place)** (например, `np.fill_diagonal(mat, 0)`).
* `np.triu_indices(n, k=1)` — возвращает индексы элементов строго выше главной диагонали (полезно для извлечения всех уникальных попарных корреляций из симметричной матрицы).

### Задача 7
Напиши функцию `process_square_matrix(M)`, которая принимает квадратную матрицу `M` размера `(N, N)` и возвращает кортеж `(trace_val, strictly_upper)`:
1. `trace_val` — след матрицы `M` (`float`).
2. `strictly_upper` — копия матрицы `M`, в которой оставлены только элементы **строго выше главной диагонали**, а сама главная диагональ и всё, что под ней, заполнено нулями 

In [48]:
def process_square_matrix(M: np.ndarray):
    # ТВОЙ КОД ЗДЕСЬ
    return (np.trace(M), np.triu(M,k=1))
    # pass


# === ТЕСТЫ ===
sq = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])
tr, upper = process_square_matrix(sq)
expected_upper = np.array([
    [0, 2, 3],
    [0, 0, 6],
    [0, 0, 0]
])
assert np.isclose(tr, 15.0), f"Неверный след матрицы (1+5+9=15): {tr}"
assert np.array_equal(upper, expected_upper), f"Неверная строго верхнетреугольная матрица:\n{upper}"
print("✅ Задача 7 решена верно!")

✅ Задача 7 решена верно!


---
## Модуль 8. Генерация случайных чисел (`np.random.default_rng`)

### 🎯 Методы, которые нужны для задачи:
* В современном NumPy рекомендуется использовать объект генератора `rng = np.random.default_rng(seed=...)` вместо устаревшего глобального `np.random.seed()`.
* Основные методы объекта `rng`:
  * `rng.normal(loc=0.0, scale=1.0, size=...)` — нормальное (гауссовское) распределение со средним `loc` и стандартным отклонением `scale`.
  * `rng.uniform(low=0.0, high=1.0, size=...)` — равномерное вещественное распределение.
  * `rng.integers(low, high, size=...)` — случайные целые числа от `low` (включительно) до `high` (не включительно).
  * `rng.permutation(n)` — возвращает случайную перестановку индексов от `0` до `n - 1` (используется для перемешивания датасета перед разбиением на train/test!).

### 📚 Связанные методы в копилку:
* `rng.choice(arr, size=k, replace=False, p=...)` — случайная выборка `k` элементов из массива (при `replace=False` — без повторений, `p` — вероятности каждого элемента).
* `rng.shuffle(arr)` — перемешивает массив **на месте (in-place)** вдоль первой оси.

### Задача 8
Напиши функцию `train_test_split_indices(n_samples, test_size, seed=42)`, которая:
1. Создает генератор `rng = np.random.default_rng(seed)`.
2. Генерирует случайную перестановку индексов от `0` до `n_samples - 1` с помощью `rng.permutation(n_samples)`.
3. Определяет число тестовых объектов `n_test = int(n_samples * test_size)`.
4. Возвращает кортеж `(train_idx, test_idx)`, где `test_idx` — это **первые** `n_test` индексов из перемешанного массива, а `train_idx` — все **остальные**.

In [49]:
def train_test_split_indices(n_samples: int, test_size: float, seed: int = 42):
    # ТВОЙ КОД ЗДЕСЬ
    rng = np.random.default_rng(seed)
    perest = rng.permutation(n_samples)
    n_test = int(n_samples * test_size)
    return (perest[n_test:], perest[:n_test])
    # pass


# === ТЕСТЫ ===
tr_idx, te_idx = train_test_split_indices(10, test_size=0.3, seed=42)
assert len(te_idx) == 3 and len(tr_idx) == 7, f"Неверные размеры выборок: {len(tr_idx)}, {len(te_idx)}"
assert len(np.intersect1d(tr_idx, te_idx)) == 0, "Индексы train и test пересекаются!"
assert np.array_equal(np.sort(np.concatenate([tr_idx, te_idx])), np.arange(10)), "Потеряны индексы!"
print("✅ Задача 8 решена верно!")

✅ Задача 8 решена верно!


---
## Модуль 9. Быстрая группировка и подсчет частот (`np.bincount`)

### 🎯 Методы, которые нужны для задачи:
* `np.bincount(x, weights=None, minlength=0)` — супер-быстрая функция для 1D-массива неотрицательных целых чисел `x`:
  * Без параметра `weights`: считает, сколько раз каждое число `i` встретилось в массиве `x` (работает в десятки раз быстрее `np.unique`, так как не делает сортировку!).
  * **С параметром `weights`:** вместо простого подсчета количества суммирует значения из массива `weights` для каждой корзины (аналог `GROUP BY x SUM(weights)` в SQL или Pandas!). Например, если `clusters = [0, 1, 0]`, а `prices = [100, 50, 200]`, то `np.bincount(clusters, weights=prices)` вернет `[300., 50.]`.

### 📚 Связанные методы в копилку:
* `np.searchsorted(sorted_arr, values)` — бинарный поиск: за $O(\log N)$ находит индексы, куда нужно вставить элементы `values` в отсортированный массив `sorted_arr`, чтобы порядок сохранился.
* `np.digitize(x, bins)` — разбивает непрерывные числа на дискретные корзины (бины).

### Задача 9
В алгоритме кластеризации K-Means нужно пересчитывать центры кластеров — то есть находить среднее значение точек в каждом кластере.

Пусть даны одномерные координаты точек `points` длины `N` и массив номеров их кластеров `cluster_ids` длины `N` (целые числа от `0` до `K - 1`, гарантируется, что в каждом кластере есть хотя бы одна точка).

Напиши функцию `compute_cluster_means(points, cluster_ids)`, которая **без циклов** с помощью двух вызовов `np.bincount` (один с `weights=points` для сумм, второй без `weights` для количества точек) возвращает 1D-массив средних значений для каждого из `K` кластеров.

In [51]:
def compute_cluster_means(points: np.ndarray, cluster_ids: np.ndarray) -> np.ndarray:
    # ТВОЙ КОД ЗДЕСЬ
    return np.bincount(cluster_ids,weights=points) / np.bincount(cluster_ids)
    # pass


# === ТЕСТЫ ===
pts = np.array([10.0, 20.0, 100.0, 30.0, 200.0])
c_ids = np.array([0, 0, 1, 0, 1])  # Кластер 0: [10, 20, 30] -> mean 20. Кластер 1: [100, 200] -> mean 150.
means = compute_cluster_means(pts, c_ids)
assert np.allclose(means, np.array([20.0, 150.0])), f"Неверные средние по кластерам: {means}"
print("✅ Задача 9 решена верно!")

✅ Задача 9 решена верно!


---
## Модуль 10. Босс-уровень с ML-собеседований: Численно стабильный Softmax

### 🎯 Формула и трюк, которые спрашивают на каждом втором собеседовании:
* Функция **Softmax** превращает вектор произвольных чисел (логитов) $z$ в вектор вероятностей (все значения $> 0$, а их сумма равна $1$):
  $$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}}$$
* **Проблема переполнения (Overflow):** если в массиве есть число `1000`, то `np.exp(1000)` вернет `inf`, а `inf / inf` превратится в `nan`!
* **Математическое решение:** если из всех элементов строки вычесть константу $C = \max(z)$, результат Softmax **не изменится** ($e^{z_i - C} / \sum e^{z_j - C} = e^{-C}e^{z_i} / (e^{-C}\sum e^{z_j})$), зато максимальная степень экспоненты станет равна $0$ ($e^0 = 1$), и переполнения вверх никогда не произойдет!

### 📚 Связанные методы в копилку:
* `np.einsum('ij,kj->ik', A, B)` — нотация Эйнштейна в NumPy: позволяет записывать любые матричные умножения, следы, диагонали и батчевые операции одной строкой.

### Задача 10
Напиши функцию `stable_softmax(logits)`, которая принимает 2D-матрицу логитов `logits` размера `(N_samples, M_classes)` и вычисляет **численно стабильный Softmax** для каждой строки (по `axis=1`):
1. Вычти из каждой строки её максимум (не забудь `axis=1, keepdims=True`).
2. Возьми экспоненту `np.exp` от сдвинутых логитов.
3. Подели каждую строку на сумму экспонент в этой строке (снова `axis=1, keepdims=True`).

In [54]:
def stable_softmax(logits: np.ndarray) -> np.ndarray:
    # ТВОЙ КОД ЗДЕСЬ
    logits = np.exp(logits - np.max(logits,axis=1,keepdims=True))
    return logits / np.sum(logits,axis=1,keepdims=True)

    # pass


# === ТЕСТЫ ===
# Проверяем на огромных числах, где обычный np.exp упал бы в inf/nan!
huge_logits = np.array([
    [1000.0, 1000.0, 1000.0],
    [2000.0, 2001.0, 2000.0]
])
probs = stable_softmax(huge_logits)
assert not np.isnan(probs).any(), "Получен NaN! Забыл вычесть максимум для стабильности?"
assert np.allclose(np.sum(probs, axis=1), np.array([1.0, 1.0])), f"Сумма вероятностей по строкам не равна 1: {np.sum(probs, axis=1)}"
assert np.allclose(probs[0], np.array([1/3, 1/3, 1/3])), f"Ошибка в первой строке: {probs[0]}"
print("🎉 ПОЗДРАВЛЯЮ! Продвинутый Уровень 2 по NumPy (10 из 10) полностью пройден!")

🎉 ПОЗДРАВЛЯЮ! Продвинутый Уровень 2 по NumPy (10 из 10) полностью пройден!
